# Dataset Overview

This notebook documents the dataset used to train and evaluate the human-vs-AI text detector in this repository. It covers:

1. **The raw text corpus** — `data/dataset_ready_final/*.jsonl`. Humans, AI-original texts, and LLM rewrites.
2. **The cached extractor features** — `data/features/*.npz`. NELA + StyleDecipher + TRACE feature triples consumed by training.
3. **How to unpack `features.tar.gz`** if you've just downloaded the built cache off a remote GPU.

For the full dataset-construction story (sources, rewrite generators, quality filtering, author-disjoint splitting), see [`README.md`](README.md). For training/evaluation, see [`../training/README.md`](../training/README.md).

## 1. The raw text corpus

The corpus combines three types of records, split **author-disjoint** into train/val/test so the TRACE author-fingerprint model cannot leak author identity across splits.

| Type | Label | Count | Description |
|---|---|---|---|
| Human texts | `human` | 3,415 | USE student essays + HC3 human answers |
| AI-original texts | `ai`    | 3,620 | ArguGPT, RAID, HC3 ChatGPT answers |
| LLM rewrites of human texts | `ai` | 6,051 | 5 LLMs paraphrasing USE essays (style-shift stress test) |
| **Total** | | **13,086** | |

| Split | Total | Human | AI |
|---|---|---|---|
| `train` | 9,097 | 2,390 | 6,707 |
| `val`   | 1,952 |   501 | 1,451 |
| `test`  | 2,037 |   524 | 1,513 |

Every record (in every `.jsonl`) shares the same schema:

| Field | Type | Meaning |
|---|---|---|
| `id` | `str` | unique ID (`h_` human, `a_` AI-original, `r_` rewrite) |
| `text` | `str` | the text content |
| `is_ai` / `label` | `bool` / `str` | redundant binary label |
| `source` | `str` | `use`, `use_rewrite`, `hc3`, `argugpt`, `raid` |
| `author_id` | `str?` | USE author (rewrites inherit their source's author) |
| `model` | `str?` | generating model for AI / rewrite records |
| `source_text_id` | `str?` | for rewrites: the source human record's `id` |
| `split` | `str` | `train` / `val` / `test` |

## 2. The cached extractor features

Running the three feature extractors (NELA, StyleDecipher, TRACE) over every record is slow — StyleDecipher in particular calls 5 LLMs per non-cluster record. To make training fast and repeatable, `training/build_dataset.py` runs the extractors **once** and caches the result. The training scripts then never touch the extractors.

The cache lives at `data/features/`:

| File | Contents |
|---|---|
| `train.npz` | feature arrays for the train split |
| `val.npz`   | feature arrays for the val split |
| `test.npz`  | feature arrays for the test split |
| `meta.json` | build configuration + per-split summary stats |

Each `.npz` contains row-aligned arrays:

| Array | Shape | Description |
|---|---|---|
| `nela`     | (N, 87)  | NELA surface / linguistic / credibility features |
| `style`    | (N, 10)  | StyleDecipher: mean + std of 5 similarity metrics vs LLM rewrites |
| `trace`    | (N, 128) | TRACE author-style embedding |
| `label`    | (N,)     | 0 = human, 1 = ai |
| `style_ok` | (N,)     | True if real StyleDecipher features were computed (False → zero vector) |
| `ids`      | (N,)     | record IDs from the source `.jsonl` (row ↔ record) |
| `sources`  | (N,)     | source corpus name (matches the `.jsonl` `source` field) |

**StyleDecipher coverage note.** `style_ok` distinguishes records that actually got LLM rewrites compared against them from ones that didn't. In `--styledecipher ollama` mode (what this cache was built with), every record's text was sent to 5 LLMs that produced fresh rewrites, so `style_ok` should be ~100 % across every source corpus.

## 3. Extract `features.tar.gz`

If you just downloaded the built cache from a remote GPU as `features.tar.gz`, the next cell unpacks it into `data/features/`. Safe to re-run — it's a no-op if `data/features/` already exists.

In [ ]:
import tarfile
from pathlib import Path

# Resolve paths relative to this notebook (works whether you launch from repo root or data/)
NB_DIR       = Path.cwd() if Path.cwd().name == "data" else Path.cwd() / "data"
REPO_ROOT    = NB_DIR.parent
FEATURES_DIR = NB_DIR / "features"
TARBALL      = REPO_ROOT / "features.tar.gz"

if FEATURES_DIR.exists() and any(FEATURES_DIR.glob("*.npz")):
    print(f"already unpacked: {FEATURES_DIR}")
elif TARBALL.exists():
    print(f"extracting {TARBALL.name} → {REPO_ROOT}")
    with tarfile.open(TARBALL) as t:
        t.extractall(REPO_ROOT)
    print(f"done  → {FEATURES_DIR}")
else:
    raise FileNotFoundError(
        f"Neither {FEATURES_DIR} nor {TARBALL} exists.\n"
        f"Either rebuild the cache (see ../training/README.md) or place features.tar.gz at the repo root."
    )

print("\nfiles in data/features/:")
for p in sorted(FEATURES_DIR.iterdir()):
    print(f"  {p.name:<14} {p.stat().st_size/1024:>8.1f} KB")

## 4. Load the cache and read the build configuration

`meta.json` records exactly how the cache was built — useful when comparing multiple cache versions.

In [ ]:
import json
import numpy as np

meta = json.loads((FEATURES_DIR / "meta.json").read_text())

print("Build configuration:")
print(f"  styledecipher_mode = {meta['styledecipher_mode']}")
print(f"  trace_context      = {meta['trace_context']}")
print(f"  dims               = {meta['dims']}")
print(f"  seed               = {meta.get('seed')}")

print("\nPer-split summary (from meta.json):")
for sp, stats in meta["splits"].items():
    print(f"  {sp:<5} records={stats['records']:>5}  human={stats['human']:>5}  ai={stats['ai']:>5}  "
          f"style_coverage={stats['style_coverage']:.1%}")

splits = {sp: np.load(FEATURES_DIR / f"{sp}.npz", allow_pickle=True) for sp in ("train", "val", "test")}
print("\nArrays per split:", list(splits["train"].files))

## 5. Per-extractor sanity check

Each row carries three feature vectors. Confirm each extractor populated real (non-zero) values across every split, and look at their numeric ranges.

In [ ]:
for sp in ("train", "val", "test"):
    d = splits[sp]
    n = len(d["ids"])
    print(f"\n=== {sp} ({n} records) ===")
    for name in ("nela", "style", "trace"):
        arr = d[name]
        nz  = int((arr != 0).any(axis=1).sum())
        print(f"  {name:<5} shape={str(arr.shape):<12} non-zero rows: {nz:>5}/{n}  "
              f"mean={arr.mean():+.4f}  std={arr.std():.4f}  min={arr.min():+.4f}  max={arr.max():+.4f}")

## 6. StyleDecipher coverage broken down by source corpus

The key health check after an `ollama`-mode build. Every source — including the non-cluster ones (`hc3`, `argugpt`, `raid`) that require fresh Ollama generations — should hit ~100 % `style_ok`. Anything materially lower means Ollama was down or rejecting requests for some records during the build.

In [ ]:
from collections import defaultdict

for sp in ("train", "val", "test"):
    d = splits[sp]
    buckets = defaultdict(list)
    for s, ok in zip(d["sources"], d["style_ok"]):
        buckets[str(s)].append(bool(ok))
    print(f"\n--- {sp} ---")
    for src in sorted(buckets):
        oks = buckets[src]
        print(f"  {src:<14} {sum(oks):>5}/{len(oks):<5} = {sum(oks)/len(oks):.0%}")

## 7. Label distribution

The dataset is intentionally imbalanced (~75 % AI, ~25 % human) because rewrites and AI-original together outnumber human texts. Training uses inverse-frequency class weighting to compensate.

In [ ]:
for sp in ("train", "val", "test"):
    labels = splits[sp]["label"]
    human  = int((labels == 0).sum())
    ai     = int((labels == 1).sum())
    total  = len(labels)
    print(f"  {sp:<5} human={human:>5} ({human/total:.1%})   ai={ai:>5} ({ai/total:.1%})   total={total}")

## 8. Where to go next

The cache is ready to feed into training and evaluation. Everything below runs locally on CPU/GPU and never touches the extractors:

```bash
# Neural fusion model — one strategy or sweep all four
python -m training.train --fusion-method gating
python -m training.train --fusion-method all

# Classical classifier — one backend or sweep all
python -m training.train_classical --classifier xgboost
python -m training.train_classical --classifier all

# Evaluate any trained checkpoint on the test split (.pt or .joblib auto-detected)
python -m test.evaluate --model models/test_models/fusion_gating.pt
```

See [`../training/README.md`](../training/README.md) for the full training framework documentation — classifier choices, hyper-parameters, fusion strategies, and per-extractor importance reporting for classical models.